# 18 — Replay programado e observabilidade local

Como tornar fila, latência, maturidade de rótulos e recuperação verificáveis?
O autor concluiu em 2026-10-15 o ensaio programado sobre eventos materializados,
com o mesmo núcleo/modelo/limiar. Não abre reserva de qualidade nem repete treino.
Os resultados abaixo são relatos de execução local, não outputs deste notebook.


## Decisões e contratos

Produtor e consumidor usam fila limitada em processo. `speed=1` preserva intervalos
da fonte; `speed>1` acelera. Cada invocação tem âncora monotônica própria. O
contexto anterior à janela é aquecido sem pacing e não entra nas latências.
O checkpoint decide o que foi confirmado. Telemetria separada marca commit sem
medição como lacuna. Duração desconhecida não vale zero.

Os scripts operacionais ficam fora do pacote congelado e têm hash próprio; os
verificadores históricos e as dependências continuam com seus bytes originais.


## O que o painel pode afirmar

| Camada | Evidência visível |
| --- | --- |
| Operação | Saúde de scrape/coleta, progresso, throughput, fila, tempos e lacunas |
| Triagem | Alertas e orçamento; precisão/recall agregados por cliente/data maduro |
| Modelo | AP das transações maduras, suporte e Jensen–Shannon dos scores |

Coortes exigem o dia completo, sete dias de atraso e todos os rótulos observados,
ou EOF completo verificado. Métricas indefinidas ficam ausentes. JS usa os últimos
1.000 scores e o desenvolvimento verificado; é diagnóstico descritivo. Features
completas, recursos nativos, drift causal e qualidade futura exigem evidência própria.


## Próximos marcos e critérios

1. Manter os checks de coleta/alerta/retorno já validados; a conferência visual dos
   painéis nativos finais ainda precisa ser informada. Não repetir o replay.
2. Inventariar Bronze/Silver/Gold, tracking, artefatos e estado; implementar S3
   compatível local (RustFS), com manifesto, cobertura e restore verificável.
3. Demonstrar CD/rollback e ampliar observabilidade de dados/features.
4. Atualização controlada (CT) com orçamento, rótulos disponíveis e gates auditáveis.
5. Provisionar em KinD + Helm, destino final escolhido, após persistência e rollback.

Um recorte 1x representativo exige protocolo e orçamento antes da execução;
não precisa repetir a janela integral em ritmo real, que levaria semanas.
Feature store depende de reutilização real, joins point-in-time, disponibilidade,
frescor online, recuperação, paridade e custo. Orquestrador e broker dependem de
necessidade demonstrada. RustFS local evita serviço cloud obrigatório; um nó/disco
não é redundância. Compose é a bancada atual para validar os componentes.


## Evidência e execução

[Runbook](../../docs/operations/OBSERVABILITY.md),
[protocolo](../../references/paced_monitoring_protocol_v1.json),
[checks da preparação](../../references/evidence/local_observability_preparation_2026-10-15.json),
[resultado nativo do autor](../../references/evidence/paced_native_author_validation_2026-10-15.json),
[infra](../../infra/observability/README.md) e
[roadmap](../../docs/project/ROADMAP.md).

A preparação usou dados gerados; o autor posteriormente executou o smoke,
confirmou a stack/painéis e concluiu o nativo. Relatos de terminal não são inspeção
direta dos bancos pelo assistente. O hash do recibo operacional foi informado,
não recalculado aqui. A coleta nativa final e o teste de exporter foram depois informados pelo autor;
[evidência de recuperação](../../references/evidence/exporter_recovery_author_validation_2026-10-15.json).
Ritmo 1x contínuo, SLA e produção permanecem sem validação.


## Resultado nativo informado pelo autor — 2026-10-15

**556.824 eventos**, sendo 278.412 transações e 278.412 feedbacks; **143.945
decisões e 1.453 alertas**, em 14 sessões a 8.640×, capacidade 128. O warmup de
201.667 eventos foi reconstruído uma vez e excluído das latências. A avaliação
independente passou: 19 features exatas, scores com `atol=1e-12`/`rtol=0`,
política causal preservada e **zero commits sem medição**.

| Tempo, em segundos | p50 | p95 | Máximo |
| --- | ---: | ---: | ---: |
| Agendado→commit | 186,449511 | 379,570051 | 423,992456 |
| Atraso de emissão | 184,698177 | 377,783443 | 422,739568 |
| Fila | 1,610370 | 3,183494 | 4,902304 |
| Processamento | 0,010024 | 0,023705 | 2,438961 |

Há 143.945 amostras em cada grupo. Retomadas usam novas âncoras; os quantis não
representam um fluxo contínuo e não devem ser somados. O processamento inclui
features, scoring e commit, não apenas inferência. A aceleração gerou atraso
anterior ao processamento; um perfil próprio seria necessário para identificar
o recurso limitante. O máximo de 2,44 s não deve desaparecer da interpretação.

A fila reproduziu **768 capturas, 358 perdas e 685 alertas sem fraude** por
cliente/data: precisão agregada 52,86%, recall 68,21%, AP transacional 0,578783
e ROC AUC 0,876322. A janela já havia sido consultada: sem confirmação nova,
intervalo de generalização, superioridade formal ou estimativa de perda evitada.

O smoke gerado demonstrou idempotência, scrape, valores dos dois dashboards e
crash/retomada com dez decisões idênticas ao contínuo. Não substitui falha nativa do replay nem verificação visual dos painéis no
volume final. O teste do exporter foi executado posteriormente, conforme abaixo.

Os três custos iniciais informados tiveram paredes de 7min09,26s, 2min13,78s e
6min51,27s, e RSS máximos de aproximadamente 341, 339 e 337 MiB. O primeiro incluiu
408,725 s de warmup. Os demais custos e os recursos Docker não foram consolidados.
O runbook foi corrigido para guardar esses arquivos fora do diretório do ensaio
antes de sua inicialização e para preservar a velocidade de 8.640× na retomada.


## Fechamento operacional e responsabilidades de armazenamento

O autor confirmou `/summary` com 556.824 eventos, 143.945 decisões, 15 dias maduros
e zero pendências/lacunas. Prometheus retornou `up=1`, coletor=1 e total 556.824.
A consulta foi corrigida para selecionar os três nomes de métricas; `or` com os
mesmos labels de correspondência havia preservado apenas a série da esquerda.

No teste real, o exporter parou, `FraudExporterUnavailable` entrou em `firing` e
o serviço retornou. Parada solicitada→alerta observado: **56,36 s**; início
solicitado→recuperação observada: **12,29 s**. Os cinco arquivos de estado/avaliação
mantiveram os hashes segundo o relatório do autor. Polling de dois segundos e
comandos Compose fazem parte desses tempos; não se validou SLA de produção.
O relatório completo e os hashes individuais não foram recebidos pelo assistente.

A rotina foi incorporada em `scripts/validation/exporter_recovery.py`, fora dos
scripts identificados do replay. Seus testes de controle usam pytest no tox
padrão; `tox -e exporter-recovery` ativa explicitamente o teste real, que fica
ignorado na suíte padrão. O resultado histórico real pertence ao utilitário anterior,
não a essa nova integração. Não é necessário executar novamente a falha ou o
replay apenas para registrar o histórico.

O [plano de armazenamento](../../docs/operations/STORAGE.md) cobre todas as camadas:
Bronze/Silver/Gold e artefatos publicados em RustFS/S3 com manifesto e restauração;
DuckDB como motor de consulta; SQLite ativo em disco persistente com backup;
PostgreSQL escolhido para o futuro servidor MLflow, com migração própria; Prometheus/Grafana com seus
volumes; feature store condicionada a reutilização e materialização demonstradas.
O piloto S3 limitado está implementado/testado com fixtures na
[etapa 19](19_storage_inventory_and_recovery.ipynb); round-trip real de dois documentos aprovado pelo autor; migração
ampla e recarga de modelo ainda pendentes. A execução começa pelo inventário, sem apagar originais,
expandindo depois a cobertura por camada. KinD + Helm continua o destino final.

[Evidência real](../../references/evidence/exporter_recovery_author_validation_2026-10-15.json),
[testes e isolamento](../../docs/operations/TESTING.md#indisponibilidade-e-recuperação-do-exporter)
e [roadmap](../../docs/project/ROADMAP.md).

O fechamento passou tox `py314`, Ruff, formatação e lockfile: 633 testes aprovados,
um teste real ignorado por padrão e 93 avisos. [Registro dos checks](../../references/evidence/observability_closure_preparation_2026-10-15.json).
